# Cardiovascular Disease Prediction on NHANES with Clustering-Augmented (Hybrid) Classifiers

**Author:** Sadia Tasnim Barsha

This notebook asks whether features learnt **without labels** (K-Means cluster membership and distances to cluster centroids) improve **supervised** prediction of cardiovascular disease (CVD) in adult NHANES participants.

Four classifiers (Logistic Regression, Random Forest, SVM, XGBoost) are trained on three feature representations:

| Representation | Features | Role |
|---|---|---|
| **Baseline** | Clinical and demographic predictors plus engineered risk features | Reference |
| **Pure Hybrid** | Baseline + `Cluster_Label` + `Distance_to_Centroid_k` | **Primary** hybrid model (no outcome information used) |
| **Augmented Hybrid** | Pure Hybrid + `Cluster_CVD_Rate` (out-of-fold target encoding) | Secondary / sensitivity analysis |

That gives **12 models** in total. All of them use the same split, the same preprocessing, the same fixed hyperparameters and the same threshold rule, so any difference between representations of the same classifier comes from the cluster features.

### Leakage controls
* The **test set (20%) is used once**, after every modelling decision has been fixed.
* Training-only steps (missing-column filtering, IQR outlier capping, imputation, scaling, K-Means, target encoding, SMOTE) are **re-fitted inside every cross-validation fold**.
* **Decision thresholds** are chosen from out-of-fold (OOF) predictions on the training set, never from the test set.
* The bootstrap comparison selects its model **by training OOF F1**, not by test performance.

### Notebook structure
1. Setup and configuration
2. Data loading and validation
3. Cleaning and feature engineering
4. Exploratory data analysis (descriptive only)
5. Train–test split and training-only filtering
6. Shared modelling functions
7. Unsupervised clustering analysis (training data only)
8. Nested cross-validation and threshold selection
9. Final held-out test evaluation (12 models)
10. Cluster stability
11. Paired bootstrap comparison
12. Results summary, interpretation and limitations

All tables are saved to `outputs/` and all figures to `outputs/figures/`.

## 1. Setup and Configuration

In [ ]:
import os
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix
from IPython.display import display

import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    silhouette_score, adjusted_rand_score,
)

try:
    import imblearn
    from imblearn.over_sampling import SMOTE
    import xgboost
    from xgboost import XGBClassifier
except ImportError as err:
    raise ImportError(
        "This notebook requires imbalanced-learn and xgboost. "
        "Install all dependencies with:  pip install -r requirements.txt"
    ) from err

print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("imblearn    ", imblearn.__version__)
print("xgboost     ", xgboost.__version__)

In [ ]:
# ------------------------------------------------------------------
# Configuration: every tunable choice of the study is defined here
# ------------------------------------------------------------------
DATA_CANDIDATES = ["data/NHANES_FINAL_DATASET.csv", "NHANES_FINAL_DATASET.csv"]

RANDOM_STATE = 42
TEST_SIZE = 0.20                 # held-out test proportion (stratified)
HIGH_MISSING_THRESHOLD = 50      # drop predictors with > 50% missing (training data)
IQR_K = 1.5                      # IQR multiplier for outlier capping

K_FINAL = 3                      # number of K-Means clusters; set to None to use the silhouette-optimal K
K_RANGE = range(2, 8)            # K values explored in the silhouette analysis
SILHOUETTE_SAMPLE = 10_000       # silhouette is O(n^2): use a random sample above this size

N_FOLDS = 5                      # cross-validation folds for OOF predictions
SMOTE_RATIO = 0.7                # minority/majority ratio after SMOTE
MIN_RECALL = 0.70                # threshold rule: maximise F1 subject to OOF recall >= this value
THRESHOLD_GRID = np.round(np.arange(0.10, 0.901, 0.01), 2)

N_STABILITY_SEEDS = 20           # K-Means restarts for the stability analysis
N_BOOTSTRAP = 1000               # bootstrap resamples of the test set

OUT_DIR = "outputs"
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


def save_fig(name):
    """Save the current figure to outputs/figures/<name>.png and display it."""
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"{name}.png"), dpi=200, bbox_inches="tight")
    plt.show()

## 2. Data Loading and Validation

The notebook expects the prepared NHANES file `NHANES_FINAL_DATASET.csv` in `data/` (or next to the notebook). It must contain a binary target column `CVD` (1 = CVD, 0 = no CVD). See `data/README.md` for the expected columns.

In [ ]:
DATA_PATH = next((p for p in DATA_CANDIDATES if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "NHANES_FINAL_DATASET.csv not found. Place it in data/ (see data/README.md)."
    )

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
df.info()

print("\nDuplicate rows:", df.duplicated().sum())
if "Participant_ID" in df.columns:
    print("Duplicate Participant_ID:", df["Participant_ID"].duplicated().sum())

if "CVD" not in df.columns:
    raise ValueError("Target column 'CVD' is missing from the dataset.")
unexpected = set(df["CVD"].dropna().unique()) - {0, 1}
if unexpected:
    raise ValueError(f"CVD must be coded 0/1; found {unexpected}")
if df["CVD"].isna().any():
    raise ValueError("CVD contains missing values; remove or recode them before modelling.")

print("\nTarget distribution:")
display(pd.DataFrame({
    "Count": df["CVD"].value_counts().sort_index(),
    "Percentage": (df["CVD"].value_counts(normalize=True).sort_index() * 100).round(2),
}))

## 3. Cleaning and Feature Engineering

The analysis is restricted to **adults (age ≥ 18)**. The engineered features below are computed **row by row** from each participant's own measurements. They use no information from other participants or from the outcome, so it is safe to create them before the train–test split.

| Feature | Definition |
|---|---|
| `Pulse_Pressure` | Systolic − diastolic BP |
| `BP_Category` | Normal / Elevated (≥120/80) / High (≥140/90) |
| `High_BP_Risk` | 1 if SBP ≥ 130 or DBP ≥ 80 |
| `BMI_Category` | Underweight / Normal / Overweight / Obese (WHO cut-offs) |
| `Age_Group` | 18–39 / 40–59 / 60+ |
| `Cholesterol_Ratio`, `Non_HDL_Cholesterol` | Total/HDL and Total − HDL cholesterol |
| `High_HbA1c_Risk` | 1 if HbA1c ≥ 6.5% (diabetes threshold) |
| Interactions | Age×BMI, Age×SBP, BMI×HbA1c, SBP×DBP |

Binary risk flags are left **missing** (not 0) when the underlying measurement is missing, so an unmeasured participant is not labelled "low risk". Missing values are imputed later inside the modelling pipeline.

In [ ]:
CODED_CATEGORICAL = [
    "Gender", "Race_Ethnicity", "Education_Level", "Diabetes_Status", "Smoking_Status",
    "BMI_Category", "Age_Group", "BP_Category", "Cluster_Label",
]
CLUSTER_FEATURES = [
    "Age", "BMI", "HbA1c", "Total_Cholesterol", "HDL_Cholesterol",
    "Average_Systolic_BP", "Average_Diastolic_BP", "Pulse_Pressure",
    "Cholesterol_Ratio", "Non_HDL_Cholesterol",
]


def _has(data, cols):
    return set(cols).issubset(data.columns)


def _flag(condition, missing):
    """Binary indicator that stays NaN where the source measurement is missing."""
    return pd.Series(np.where(missing, np.nan, condition.astype(float)), index=condition.index)


def engineer_features(data):
    d = data.copy()
    sbp, dbp = "Average_Systolic_BP", "Average_Diastolic_BP"

    if _has(d, [sbp, dbp]):
        bp_missing = d[sbp].isna() | d[dbp].isna()
        d["Pulse_Pressure"] = d[sbp] - d[dbp]
        bp_cat = np.select(
            [(d[sbp] >= 140) | (d[dbp] >= 90), (d[sbp] >= 120) | (d[dbp] >= 80)],
            ["High", "Elevated"], default="Normal",
        ).astype(object)
        bp_cat[bp_missing.to_numpy()] = np.nan
        d["BP_Category"] = bp_cat
        d["High_BP_Risk"] = _flag((d[sbp] >= 130) | (d[dbp] >= 80), bp_missing)
        d["SBP_x_DBP"] = d[sbp] * d[dbp]

    if "BMI" in d.columns:
        d["BMI_Category"] = pd.cut(d["BMI"], bins=[0, 18.5, 25, 30, np.inf],
                                   labels=["Underweight", "Normal", "Overweight", "Obese"], right=False)
    if "Age" in d.columns:
        d["Age_Group"] = pd.cut(d["Age"], bins=[17, 39, 59, np.inf], labels=["18-39", "40-59", "60+"])

    if _has(d, ["Total_Cholesterol", "HDL_Cholesterol"]):
        d["Cholesterol_Ratio"] = d["Total_Cholesterol"] / d["HDL_Cholesterol"].replace(0, np.nan)
        d["Non_HDL_Cholesterol"] = d["Total_Cholesterol"] - d["HDL_Cholesterol"]

    if "HbA1c" in d.columns:
        d["High_HbA1c_Risk"] = _flag(d["HbA1c"] >= 6.5, d["HbA1c"].isna())

    if _has(d, ["Age", "BMI"]):
        d["Age_x_BMI"] = d["Age"] * d["BMI"]
    if _has(d, ["Age", sbp]):
        d["Age_x_Systolic_BP"] = d["Age"] * d[sbp]
    if _has(d, ["BMI", "HbA1c"]):
        d["BMI_x_HbA1c"] = d["BMI"] * d["HbA1c"]

    return d.replace([np.inf, -np.inf], np.nan)


adult_df = df[df["Age"] >= 18].copy() if "Age" in df.columns else df.copy()
print("Original shape:  ", df.shape)
print("Adult-only shape:", adult_df.shape)

clean_df = engineer_features(adult_df)
clean_df = clean_df[[c for c in clean_df.columns if c != "CVD"] + ["CVD"]]
print("After feature engineering:", clean_df.shape)
display(clean_df.head())

## 4. Exploratory Data Analysis

This section is **descriptive only**. It summarises the adult sample, and no modelling decision is taken from it. Predictors with heavy missingness are removed later, using the training data only (Section 5).

In [ ]:
missing_table = pd.DataFrame({
    "Missing_Count": clean_df.isnull().sum(),
    "Missing_Percentage": (clean_df.isnull().mean() * 100).round(2),
}).sort_values("Missing_Percentage", ascending=False)
display(missing_table[missing_table["Missing_Count"] > 0])

plot_missing = missing_table[missing_table["Missing_Percentage"] > 0]
if len(plot_missing):
    plt.figure(figsize=(12, 5))
    plot_missing["Missing_Percentage"].plot(kind="bar", color="#4b5563")
    plt.axhline(HIGH_MISSING_THRESHOLD, ls="--", color="#dc2626", label=f"{HIGH_MISSING_THRESHOLD}% cut-off")
    plt.title("Missing Values by Feature (adult sample)")
    plt.ylabel("Missing (%)")
    plt.legend()
    save_fig("eda_missing_values")

In [ ]:
counts = clean_df["CVD"].value_counts().sort_index()
plt.figure(figsize=(5, 4))
plt.bar(["No CVD (0)", "CVD (1)"], counts.values, color=["#6b7280", "#dc2626"])
for i, v in enumerate(counts.values):
    plt.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.title("CVD Class Distribution (adults)")
plt.ylabel("Participants")
save_fig("eda_class_distribution")
print(f"CVD prevalence: {clean_df['CVD'].mean() * 100:.2f}%")

In [ ]:
numerical_cols = [c for c in clean_df.columns
                  if pd.api.types.is_numeric_dtype(clean_df[c])
                  and c not in ["Participant_ID", "CVD"] + CODED_CATEGORICAL]
print("Numerical predictors:", len(numerical_cols))
display(clean_df[numerical_cols].describe().T.round(2))

In [ ]:
KEY_CLINICAL = [c for c in ["Age", "BMI", "Average_Systolic_BP", "Average_Diastolic_BP",
                            "HDL_Cholesterol", "Total_Cholesterol", "HbA1c", "Pulse_Pressure"]
                if c in clean_df.columns]

# Histograms
n = len(KEY_CLINICAL)
fig, axes = plt.subplots(int(np.ceil(n / 4)), 4, figsize=(16, 3.5 * int(np.ceil(n / 4))))
for ax, col in zip(np.ravel(axes), KEY_CLINICAL):
    clean_df[col].dropna().hist(bins=30, ax=ax, color="#0f766e")
    ax.set_title(col)
for ax in np.ravel(axes)[n:]:
    ax.set_visible(False)
fig.suptitle("Distributions of Key Clinical Variables", y=1.02)
save_fig("eda_histograms")

# Boxplots by CVD status
fig, axes = plt.subplots(int(np.ceil(n / 4)), 4, figsize=(16, 3.8 * int(np.ceil(n / 4))))
for ax, col in zip(np.ravel(axes), KEY_CLINICAL):
    clean_df.boxplot(column=col, by="CVD", ax=ax)
    ax.set_title(col)
    ax.set_xlabel("CVD")
for ax in np.ravel(axes)[n:]:
    ax.set_visible(False)
fig.suptitle("Key Clinical Variables by CVD Status", y=1.02)
save_fig("eda_boxplots_by_cvd")

In [ ]:
grouped_means = clean_df.groupby("CVD")[KEY_CLINICAL].mean().T
grouped_means.columns = ["No_CVD_Mean", "CVD_Mean"]
grouped_means["Difference"] = grouped_means["CVD_Mean"] - grouped_means["No_CVD_Mean"]
display(grouped_means.round(2))

grouped_means[["No_CVD_Mean", "CVD_Mean"]].plot(kind="bar", figsize=(12, 5), color=["#6b7280", "#dc2626"])
plt.title("Mean Clinical Values by CVD Status")
plt.ylabel("Mean value")
plt.xticks(rotation=30, ha="right")
save_fig("eda_means_by_cvd")

In [ ]:
eda_categorical = [c for c in clean_df.columns
                   if c not in ["CVD", "Participant_ID"]
                   and (c in CODED_CATEGORICAL or not pd.api.types.is_numeric_dtype(clean_df[c]))]
print("Categorical variables:", eda_categorical)

n = len(eda_categorical)
if n:
    fig, axes = plt.subplots(int(np.ceil(n / 3)), 3, figsize=(15, 3.8 * int(np.ceil(n / 3))))
    for ax, col in zip(np.ravel(axes), eda_categorical):
        rate = clean_df.groupby(col, observed=True)["CVD"].agg(["mean", "size"])
        rate["CVD_Rate_%"] = (rate["mean"] * 100).round(2)
        print(f"\nCVD rate by {col}")
        display(rate[["size", "CVD_Rate_%"]].rename(columns={"size": "N"}))
        rate["CVD_Rate_%"].plot(kind="bar", ax=ax, color="#7c3aed")
        ax.set_title(f"CVD rate by {col}")
        ax.set_ylabel("CVD rate (%)")
        ax.tick_params(axis="x", rotation=30)
    for ax in np.ravel(axes)[n:]:
        ax.set_visible(False)
    save_fig("eda_cvd_rate_by_category")

In [ ]:
corr = clean_df[numerical_cols + ["CVD"]].corr()
print("Correlation with CVD (Pearson / point-biserial):")
display(corr["CVD"].drop("CVD").sort_values(ascending=False).to_frame("r").round(3))

plt.figure(figsize=(12, 10))
plt.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr)), corr.columns, rotation=90)
plt.yticks(range(len(corr)), corr.index)
plt.title("Correlation Heatmap")
plt.grid(False)
save_fig("eda_correlation_heatmap")

In [ ]:
scatter_cols = [c for c in ["Age", "BMI", "Average_Systolic_BP", "HbA1c"] if c in clean_df.columns]
if len(scatter_cols) >= 2:
    sample = clean_df[scatter_cols + ["CVD"]].dropna()
    sample = sample.sample(min(len(sample), 3000), random_state=RANDOM_STATE)
    colours = np.where(sample["CVD"] == 1, "#dc2626", "#9ca3af")
    scatter_matrix(sample[scatter_cols], figsize=(10, 10), diagonal="hist", c=colours, alpha=0.5)
    plt.suptitle("Scatter Matrix (red = CVD, grey = no CVD; random sample of up to 3,000)")
    save_fig("eda_scatter_matrix")

## 5. Train–Test Split and Training-Only Filtering

A stratified 80/20 split is made **before** any data-driven preprocessing. The held-out test set is not used again until Section 9.

Predictors with more than 50% missing values are identified **on the training set only** and dropped from both sets. This decision does not use the outcome.

In [ ]:
X_all = clean_df.drop(columns=["CVD", "Participant_ID"], errors="ignore")
y_all = clean_df["CVD"].astype(int)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all
)
X_train_raw, X_test_raw = X_train_raw.reset_index(drop=True), X_test_raw.reset_index(drop=True)
y_train, y_test = y_train.reset_index(drop=True), y_test.reset_index(drop=True)

train_missing_pct = X_train_raw.isna().mean() * 100
HIGH_MISSING_COLS = train_missing_pct[train_missing_pct > HIGH_MISSING_THRESHOLD].index.tolist()
X_train_raw = X_train_raw.drop(columns=HIGH_MISSING_COLS)
X_test_raw = X_test_raw.drop(columns=HIGH_MISSING_COLS)
CLUSTER_FEATURES = [c for c in CLUSTER_FEATURES if c in X_train_raw.columns]

print("Dropped (>50% missing in training data):", HIGH_MISSING_COLS or "none")
print("Training set:", X_train_raw.shape, f"| CVD prevalence {y_train.mean() * 100:.2f}%")
print("Test set:    ", X_test_raw.shape, f"| CVD prevalence {y_test.mean() * 100:.2f}%")
print("Clustering features:", CLUSTER_FEATURES)

## 6. Shared Modelling Functions

Every step below is defined **once** and reused by both the cross-validation loop and the final test evaluation, so the two use identical code.

**Outlier capping.** IQR bounds (Q1 − 1.5·IQR, Q3 + 1.5·IQR) are learnt from the data each model is fitted on. Columns with two or fewer distinct values (binary flags) and columns with zero IQR are **not capped**: capping them would set every value to a constant and erase the feature.

**Cluster features.** K-Means is fitted on the imputed and standardised clinical features listed above. The *Pure Hybrid* adds the cluster label (one-hot encoded) and the Euclidean distance to each centroid. Cluster size is **not** used, because it is a one-to-one function of the cluster label and adds no information.

**Target-encoded cluster rate (Augmented Hybrid only).** `Cluster_CVD_Rate` is the CVD prevalence of the participant's cluster. Training rows receive **out-of-fold** rates, so a participant's own outcome never contributes to their feature value. Evaluation rows receive rates computed from the full fitting set.

**Class imbalance.** Each classifier uses exactly **one** imbalance strategy, so the correction is not applied twice:

| Classifier | Strategy |
|---|---|
| Logistic Regression | SMOTE (minority:majority = 0.7), applied to training folds only |
| Random Forest | SMOTE (minority:majority = 0.7), applied to training folds only |
| SVM (RBF) | `class_weight="balanced"` (SMOTE with Platt scaling is very slow for SVM) |
| XGBoost | `scale_pos_weight` = negatives / positives |

**Hyperparameters** are fixed in advance and identical across the three representations. This keeps the baseline-vs-hybrid comparison fair; it is not an attempt to maximise absolute performance.

**Threshold rule.** On OOF training predictions, choose the threshold (0.10–0.90) that maximises F1 among thresholds with recall ≥ 0.70. If none reaches that recall, choose the threshold that maximises F1.

In [ ]:
CLASSIFIERS = ["Logistic Regression", "Random Forest", "SVM", "XGBoost"]
VERSIONS = ["Baseline", "Pure Hybrid", "Augmented Hybrid"]
IMBALANCE_STRATEGY = {
    "Logistic Regression": "SMOTE",
    "Random Forest": "SMOTE",
    "SVM": "class_weight",
    "XGBoost": "scale_pos_weight",
}


# ---------- feature typing & preprocessing ----------
def get_feature_groups(X):
    numeric = [c for c in X.columns
               if pd.api.types.is_numeric_dtype(X[c]) and c not in CODED_CATEGORICAL]
    categorical = [c for c in X.columns if c not in numeric]
    return numeric, categorical


def build_preprocessor(X):
    numeric, categorical = get_feature_groups(X)
    return ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), numeric),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
         categorical),
    ])


# ---------- IQR outlier capping ----------
def fit_iqr_bounds(X, k=IQR_K):
    numeric, _ = get_feature_groups(X)
    bounds = {}
    for c in numeric:
        s = X[c].dropna()
        if s.nunique() <= 2:          # binary flags: never cap
            continue
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr == 0:                  # degenerate spread: capping would erase the feature
            continue
        bounds[c] = (q1 - k * iqr, q3 + k * iqr)
    return bounds


def apply_iqr_bounds(X, bounds):
    out = X.copy()
    for c, (lo, hi) in bounds.items():
        if c in out.columns:
            out[c] = out[c].clip(lo, hi)
    return out


# ---------- clustering features ----------
def make_cluster_scaler():
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])


def fit_kmeans(X_fit, k, seed):
    scaler = make_cluster_scaler()
    Z = scaler.fit_transform(X_fit[CLUSTER_FEATURES])
    km = KMeans(n_clusters=k, random_state=seed, n_init=20).fit(Z)
    return scaler, km


def add_pure_cluster_features(X, scaler, km):
    Z = scaler.transform(X[CLUSTER_FEATURES])
    out = X.copy()
    out["Cluster_Label"] = km.predict(Z).astype(str)
    dist = km.transform(Z)
    for j in range(km.n_clusters):
        out[f"Distance_to_Centroid_{j}"] = dist[:, j]
    return out


def oof_cluster_rate(labels, y, seed, n_splits=N_FOLDS):
    labels, y = np.asarray(labels), np.asarray(y)
    rates = np.zeros(len(y), dtype=float)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr, va in cv.split(np.zeros(len(y)), y):
        mapping = pd.Series(y[tr]).groupby(labels[tr]).mean()
        rates[va] = pd.Series(labels[va]).map(mapping).fillna(y[tr].mean()).to_numpy()
    return rates


def add_augmented_feature(pure_fit, pure_eval, y_fit, seed):
    y_fit = np.asarray(y_fit)
    a, b = pure_fit.copy(), pure_eval.copy()
    a["Cluster_CVD_Rate"] = oof_cluster_rate(a["Cluster_Label"].to_numpy(), y_fit, seed)
    mapping = pd.Series(y_fit).groupby(a["Cluster_Label"].to_numpy()).mean()
    b["Cluster_CVD_Rate"] = b["Cluster_Label"].map(mapping).fillna(y_fit.mean()).astype(float)
    return a, b


def build_representations(X_fit, X_eval, y_fit, seed):
    """Fit every data-driven step on X_fit only; return the 3 representations for (fit, eval)."""
    bounds = fit_iqr_bounds(X_fit)
    Xf, Xe = apply_iqr_bounds(X_fit, bounds), apply_iqr_bounds(X_eval, bounds)
    scaler, km = fit_kmeans(Xf, K_FINAL, seed)
    pf, pe = add_pure_cluster_features(Xf, scaler, km), add_pure_cluster_features(Xe, scaler, km)
    af, ae = add_augmented_feature(pf, pe, y_fit, seed + 1000)
    reps = {"Baseline": (Xf, Xe), "Pure Hybrid": (pf, pe), "Augmented Hybrid": (af, ae)}
    return reps, {"bounds": bounds, "scaler": scaler, "kmeans": km}


# ---------- classifiers ----------
def make_classifier(name, y_fit, seed):
    y_fit = np.asarray(y_fit)
    if name == "Logistic Regression":
        return LogisticRegression(max_iter=3000, random_state=seed)
    if name == "Random Forest":
        return RandomForestClassifier(n_estimators=500, max_depth=8, min_samples_leaf=2,
                                      random_state=seed, n_jobs=-1)
    if name == "SVM":
        return SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced",
                   probability=True, cache_size=2000, random_state=seed)
    if name == "XGBoost":
        spw = (y_fit == 0).sum() / max((y_fit == 1).sum(), 1)
        return XGBClassifier(n_estimators=400, max_depth=3, learning_rate=0.05, subsample=0.85,
                             colsample_bytree=0.85, scale_pos_weight=spw, eval_metric="logloss",
                             random_state=seed, n_jobs=-1)
    raise ValueError(name)


def fit_and_score(name, X_fit, y_fit, X_eval, seed):
    """Preprocess, rebalance (if SMOTE), fit, and return P(CVD=1) for X_eval."""
    pre = build_preprocessor(X_fit)
    A, B = pre.fit_transform(X_fit), pre.transform(X_eval)
    y = np.asarray(y_fit)
    if IMBALANCE_STRATEGY[name] == "SMOTE":
        A, y = SMOTE(sampling_strategy=SMOTE_RATIO, random_state=seed).fit_resample(A, y)
    model = make_classifier(name, y_fit, seed).fit(A, y)
    return model.predict_proba(B)[:, 1]


# ---------- thresholds & metrics ----------
def select_threshold(y_true, score, min_recall=MIN_RECALL):
    y_true = np.asarray(y_true)
    rows = []
    for t in THRESHOLD_GRID:
        p = (score >= t).astype(int)
        rows.append({"Threshold": t,
                     "F1": f1_score(y_true, p, zero_division=0),
                     "Recall": recall_score(y_true, p, zero_division=0),
                     "Precision": precision_score(y_true, p, zero_division=0)})
    table = pd.DataFrame(rows)
    eligible = table[table["Recall"] >= min_recall]
    pool = eligible if len(eligible) else table
    best = pool.sort_values(["F1", "Precision"], ascending=False).iloc[0]
    return float(best["Threshold"]), table


def compute_metrics(y_true, score, threshold):
    pred = (score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall_Sensitivity": recall_score(y_true, pred, zero_division=0),
        "Specificity": tn / (tn + fp) if (tn + fp) else 0.0,
        "F1_Score": f1_score(y_true, pred, zero_division=0),
        "AUROC": roc_auc_score(y_true, score),
        "AUPRC": average_precision_score(y_true, score),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }


def silhouette(Z, labels, seed=RANDOM_STATE):
    size = SILHOUETTE_SAMPLE if len(Z) > SILHOUETTE_SAMPLE else None
    return silhouette_score(Z, labels, sample_size=size, random_state=seed)


def describe_silhouette(s):
    """Kaufman & Rousseeuw (1990) rule of thumb."""
    if s > 0.70: return "strong structure"
    if s > 0.50: return "reasonable structure"
    if s > 0.25: return "weak structure that could be artificial"
    return "no substantial cluster structure (clusters overlap heavily)"


print("Shared modelling functions defined.")

## 7. Unsupervised Clustering Analysis (Training Data Only)

This section describes the clusters themselves. K-Means is fitted on the **training set only**, after training-based IQR capping, imputation and scaling. No outcome information is used to form the clusters. CVD rates are shown afterwards, only to describe the clusters.

The number of clusters is examined with the silhouette coefficient for K = 2–7. `K_FINAL` is fixed in the configuration cell (default 3, chosen for clinical interpretability). Set it to `None` to use the silhouette-optimal K instead.

In [ ]:
explore_bounds = fit_iqr_bounds(X_train_raw)
X_train_capped = apply_iqr_bounds(X_train_raw, explore_bounds)
cluster_scaler = make_cluster_scaler()
Z_train = cluster_scaler.fit_transform(X_train_capped[CLUSTER_FEATURES])

sil_rows = []
for k in K_RANGE:
    labels_k = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20).fit_predict(Z_train)
    sil_rows.append({"K": k, "Silhouette": silhouette(Z_train, labels_k)})
silhouette_df = pd.DataFrame(sil_rows)
display(silhouette_df.round(4))

best_k = int(silhouette_df.loc[silhouette_df["Silhouette"].idxmax(), "K"])
if K_FINAL is None:
    K_FINAL = best_k
print(f"Silhouette-optimal K = {best_k}; K used for all hybrid models: K_FINAL = {K_FINAL}")
if K_FINAL != best_k:
    print("Note: K_FINAL differs from the silhouette-optimal K; justify this choice in the write-up.")

plt.figure(figsize=(6, 4))
plt.plot(silhouette_df["K"], silhouette_df["Silhouette"], marker="o", color="#0f766e")
plt.axvline(K_FINAL, ls="--", color="#6b7280", label=f"K_FINAL = {K_FINAL}")
plt.title("K-Means Silhouette Score by K (training data)")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Silhouette score")
plt.legend()
save_fig("cluster_silhouette_by_k")

In [ ]:
kmeans_explore = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=20).fit(Z_train)
train_clusters = kmeans_explore.labels_
SILHOUETTE_K_FINAL = silhouette(Z_train, train_clusters)
print(f"Silhouette at K = {K_FINAL}: {SILHOUETTE_K_FINAL:.3f} -> {describe_silhouette(SILHOUETTE_K_FINAL)}")

pca = PCA(n_components=2, random_state=RANDOM_STATE)
Z_pca = pca.fit_transform(Z_train)
plt.figure(figsize=(7, 5))
sc = plt.scatter(Z_pca[:, 0], Z_pca[:, 1], c=train_clusters, cmap="viridis", s=8, alpha=0.5)
plt.legend(*sc.legend_elements(), title="Cluster")
plt.title(f"K-Means Clusters (K = {K_FINAL}) Projected onto 2 Principal Components")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}% variance)")
save_fig("cluster_pca_projection")

In [ ]:
profile = X_train_capped[CLUSTER_FEATURES].copy()
profile["Cluster"] = train_clusters
profile["CVD"] = y_train.to_numpy()

cluster_summary = profile.groupby("Cluster").agg(N=("CVD", "size"), CVD_Rate_pct=("CVD", "mean"))
cluster_summary["CVD_Rate_pct"] = (cluster_summary["CVD_Rate_pct"] * 100).round(2)
cluster_summary["Share_pct"] = (cluster_summary["N"] / len(profile) * 100).round(1)
print("Cluster sizes and CVD prevalence (training set):")
display(cluster_summary)

cluster_profile = profile.groupby("Cluster")[CLUSTER_FEATURES].mean()
print("Mean clinical profile by cluster (after IQR capping):")
display(cluster_profile.round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
cluster_summary["CVD_Rate_pct"].plot(kind="bar", ax=axes[0], color="#dc2626")
axes[0].axhline(y_train.mean() * 100, ls="--", color="black", label="Overall prevalence")
axes[0].set_title("CVD Prevalence by Cluster")
axes[0].set_ylabel("CVD rate (%)")
axes[0].legend()
axes[0].tick_params(axis="x", rotation=0)

z_profile = (cluster_profile - profile[CLUSTER_FEATURES].mean()) / profile[CLUSTER_FEATURES].std()
im = axes[1].imshow(z_profile.T, cmap="RdBu_r", vmin=-1.5, vmax=1.5, aspect="auto")
axes[1].set_xticks(range(K_FINAL), [f"Cluster {i}" for i in range(K_FINAL)])
axes[1].set_yticks(range(len(CLUSTER_FEATURES)), CLUSTER_FEATURES)
axes[1].set_title("Cluster Profiles (standardised means)")
axes[1].grid(False)
plt.colorbar(im, ax=axes[1], label="z-score")
save_fig("cluster_profiles")

## 8. Nested Cross-Validation and Threshold Selection

For each of the 5 training folds:

1. IQR bounds, imputation, scaling and K-Means are fitted on the four training folds only.
2. The three representations are built for the training folds and for the held-out fold.
3. All four classifiers are trained, and they predict probabilities for the held-out fold.

This produces **out-of-fold probabilities for every training participant** under all 12 model configurations, and these are used to select each model's decision threshold. The cluster features are rebuilt inside each fold, so the OOF estimates are not optimistically biased by clustering on data the fold model later predicts.

> **Runtime:** SVM with `probability=True` runs an internal cross-validation for Platt scaling, so this cell can take several minutes.

In [ ]:
oof = {(c, v): np.full(len(y_train), np.nan) for c in CLASSIFIERS for v in VERSIONS}
outer_cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print(f"Running {N_FOLDS}-fold nested OOF analysis for {len(CLASSIFIERS) * len(VERSIONS)} models...", flush=True)
for fold, (tr, va) in enumerate(outer_cv.split(X_train_raw, y_train), start=1):
    t0 = time.time()
    X_fit = X_train_raw.iloc[tr].reset_index(drop=True)
    X_val = X_train_raw.iloc[va].reset_index(drop=True)
    y_fit = y_train.iloc[tr].to_numpy()
    reps, _ = build_representations(X_fit, X_val, y_fit, seed=RANDOM_STATE + fold)
    for version, (A, B) in reps.items():
        for clf in CLASSIFIERS:
            oof[(clf, version)][va] = fit_and_score(clf, A, y_fit, B, seed=RANDOM_STATE + fold)
    print(f"  fold {fold}/{N_FOLDS} done in {time.time() - t0:.1f}s", flush=True)

assert not any(np.isnan(v).any() for v in oof.values()), "Missing OOF predictions"

In [ ]:
thr_rows, thr_tables = [], {}
for clf in CLASSIFIERS:
    for version in VERSIONS:
        t, table = select_threshold(y_train, oof[(clf, version)])
        thr_tables[(clf, version)] = table
        m = compute_metrics(y_train, oof[(clf, version)], t)
        thr_rows.append({"Classifier": clf, "Version": version, "Threshold": t,
                         "OOF_F1": m["F1_Score"], "OOF_Recall": m["Recall_Sensitivity"],
                         "OOF_Precision": m["Precision"], "OOF_AUROC": m["AUROC"], "OOF_AUPRC": m["AUPRC"]})
oof_results = pd.DataFrame(thr_rows)
oof_results.to_csv(os.path.join(OUT_DIR, "oof_thresholds_and_metrics.csv"), index=False)

print("Training OOF performance at the selected thresholds (no test data used):")
display(oof_results.round(3))

## 9. Final Held-Out Test Evaluation (12 Models)

Every data-driven step is refitted on the **complete training set**. Each model is then applied **once** to the test set, using the threshold fixed in Section 8.

`results_all_12_models.csv` is the single source of truth for the final metrics reported in the write-up.

In [ ]:
final_reps, final_artifacts = build_representations(X_train_raw, X_test_raw, y_train.to_numpy(), seed=RANDOM_STATE)

rows, test_scores, test_preds = [], {}, {}
for version, (A, B) in final_reps.items():
    for clf in CLASSIFIERS:
        score = fit_and_score(clf, A, y_train, B, seed=RANDOM_STATE)
        t = float(oof_results.query("Classifier == @clf and Version == @version")["Threshold"].iloc[0])
        m = compute_metrics(y_test, score, t)
        rows.append({"Classifier": clf, "Version": version, "Threshold": t, **m})
        test_scores[(clf, version)] = score
        test_preds[(clf, version)] = (score >= t).astype(int)

results = pd.DataFrame(rows)
results["Classifier"] = pd.Categorical(results["Classifier"], CLASSIFIERS, ordered=True)
results["Version"] = pd.Categorical(results["Version"], VERSIONS, ordered=True)
results = results.sort_values(["Classifier", "Version"]).reset_index(drop=True)
results.to_csv(os.path.join(OUT_DIR, "results_all_12_models.csv"), index=False)

print("FINAL TEST-SET RESULTS")
display(results.round(3))

In [ ]:
METRICS = ["Accuracy", "Precision", "Recall_Sensitivity", "Specificity", "F1_Score", "AUROC", "AUPRC"]

comp_rows = []
for clf in CLASSIFIERS:
    base = results[(results.Classifier == clf) & (results.Version == "Baseline")].iloc[0]
    for version in ["Pure Hybrid", "Augmented Hybrid"]:
        hyb = results[(results.Classifier == clf) & (results.Version == version)].iloc[0]
        row = {"Classifier": clf, "Comparison": f"{version} - Baseline"}
        for met in METRICS + ["FP", "FN"]:
            row[f"Delta_{met}"] = hyb[met] - base[met]
        comp_rows.append(row)
comparison = pd.DataFrame(comp_rows)
comparison.to_csv(os.path.join(OUT_DIR, "hybrid_minus_baseline_deltas.csv"), index=False)
print("Change versus the matched baseline (positive = higher than baseline; for FP/FN, negative is better):")
display(comparison.round(3))

In [ ]:
# Grouped metric comparison
plot_metrics = ["F1_Score", "Recall_Sensitivity", "Specificity", "AUROC", "AUPRC"]
colours = ["#6b7280", "#0f766e", "#7c3aed"]
fig, axes = plt.subplots(1, len(plot_metrics), figsize=(22, 4.5), sharey=True)
for ax, met in zip(axes, plot_metrics):
    piv = results.pivot(index="Classifier", columns="Version", values=met).loc[CLASSIFIERS, VERSIONS]
    piv.plot(kind="bar", ax=ax, color=colours, legend=(met == plot_metrics[0]), width=0.8)
    ax.set_title(met)
    ax.set_xlabel("")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=30)
fig.suptitle("Held-Out Test Performance: Baseline vs Pure Hybrid vs Augmented Hybrid", y=1.03)
save_fig("results_metric_comparison")

# Pure Hybrid minus Baseline (percentage points)
delta_metrics = ["Accuracy", "Recall_Sensitivity", "Specificity", "F1_Score", "AUROC", "AUPRC"]
pure_delta = comparison[comparison.Comparison == "Pure Hybrid - Baseline"].set_index("Classifier")
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, met in zip(axes.ravel(), delta_metrics):
    vals = pure_delta[f"Delta_{met}"].loc[CLASSIFIERS] * 100
    ax.bar(CLASSIFIERS, vals, color=["#0f766e" if v >= 0 else "#dc2626" for v in vals])
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"Δ {met} (pp)")
    ax.tick_params(axis="x", rotation=30)
fig.suptitle("Pure Hybrid minus Matched Baseline (test set)", y=1.02)
save_fig("results_pure_minus_baseline")

In [ ]:
# Confusion matrices for all 12 models
fig, axes = plt.subplots(len(VERSIONS), len(CLASSIFIERS), figsize=(16, 11))
for i, version in enumerate(VERSIONS):
    for j, clf in enumerate(CLASSIFIERS):
        ax = axes[i, j]
        cm = confusion_matrix(y_test, test_preds[(clf, version)], labels=[0, 1])
        ax.imshow(cm, cmap="Blues")
        for a in range(2):
            for b in range(2):
                ax.text(b, a, f"{cm[a, b]:,}", ha="center", va="center",
                        color="white" if cm[a, b] > cm.max() / 2 else "black")
        ax.set_xticks([0, 1], ["No CVD", "CVD"])
        ax.set_yticks([0, 1], ["No CVD", "CVD"])
        ax.set_title(f"{clf}\n{version}", fontsize=10)
        ax.grid(False)
        if j == 0: ax.set_ylabel("Actual")
        if i == len(VERSIONS) - 1: ax.set_xlabel("Predicted")
save_fig("results_confusion_matrices")

# ROC and precision-recall curves: Baseline vs Pure Hybrid
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
palette = dict(zip(CLASSIFIERS, ["#2563eb", "#16a34a", "#d97706", "#7c3aed"]))
for clf in CLASSIFIERS:
    for version, ls in [("Baseline", "--"), ("Pure Hybrid", "-")]:
        s = test_scores[(clf, version)]
        fpr, tpr, _ = roc_curve(y_test, s)
        axes[0].plot(fpr, tpr, ls=ls, color=palette[clf], label=f"{clf} – {version} ({roc_auc_score(y_test, s):.3f})")
        prec, rec, _ = precision_recall_curve(y_test, s)
        axes[1].plot(rec, prec, ls=ls, color=palette[clf], label=f"{clf} – {version} ({average_precision_score(y_test, s):.3f})")
axes[0].plot([0, 1], [0, 1], ":", color="gray")
axes[0].set(title="ROC Curves (dashed = Baseline, solid = Pure Hybrid)", xlabel="False positive rate", ylabel="Sensitivity")
axes[1].axhline(y_test.mean(), ls=":", color="gray", label=f"Prevalence ({y_test.mean():.3f})")
axes[1].set(title="Precision–Recall Curves", xlabel="Recall", ylabel="Precision")
axes[0].legend(fontsize=7)
axes[1].legend(fontsize=7)
save_fig("results_roc_pr_curves")

## 10. Cluster Stability

The hybrid features are only meaningful if the clusters are reproducible. K-Means (K = `K_FINAL`) is re-run on the full, capped training set with 20 different random seeds:

* **Silhouette** shows how well separated the clusters are (Kaufman & Rousseeuw: ≤ 0.25 no substantial structure; 0.26–0.50 weak; 0.51–0.70 reasonable; > 0.70 strong).
* **Pairwise Adjusted Rand Index (ARI)** shows whether different seeds produce the same partition (1 = identical).

In [ ]:
final_bounds = final_artifacts["bounds"]
Z_stab = make_cluster_scaler().fit_transform(apply_iqr_bounds(X_train_raw, final_bounds)[CLUSTER_FEATURES])

stab_labels, stab_rows = [], []
for seed in range(N_STABILITY_SEEDS):
    lab = KMeans(n_clusters=K_FINAL, random_state=seed, n_init=20).fit_predict(Z_stab)
    stab_labels.append(lab)
    stab_rows.append({"Seed": seed, "Silhouette": silhouette(Z_stab, lab)})
stability = pd.DataFrame(stab_rows)
aris = [adjusted_rand_score(stab_labels[i], stab_labels[j])
        for i in range(N_STABILITY_SEEDS) for j in range(i + 1, N_STABILITY_SEEDS)]

stability_summary = pd.DataFrame([{
    "K": K_FINAL,
    "Mean_Silhouette": stability.Silhouette.mean(), "SD_Silhouette": stability.Silhouette.std(ddof=1),
    "Mean_Pairwise_ARI": np.mean(aris), "SD_Pairwise_ARI": np.std(aris, ddof=1), "Min_Pairwise_ARI": np.min(aris),
}])
stability.to_csv(os.path.join(OUT_DIR, "cluster_stability_by_seed.csv"), index=False)
stability_summary.to_csv(os.path.join(OUT_DIR, "cluster_stability_summary.csv"), index=False)
display(stability_summary.round(4))
print("Interpretation:", describe_silhouette(stability.Silhouette.mean()))

plt.figure(figsize=(8, 4.5))
plt.plot(stability.Seed, stability.Silhouette, marker="o", color="#0f766e")
plt.axhline(stability.Silhouette.mean(), ls="--", color="#dc2626", label=f"Mean = {stability.Silhouette.mean():.3f}")
plt.title(f"K = {K_FINAL} Silhouette Across {N_STABILITY_SEEDS} Random Seeds (mean pairwise ARI = {np.mean(aris):.3f})")
plt.xlabel("Random seed")
plt.ylabel("Silhouette score")
plt.legend()
save_fig("cluster_stability")

## 11. Paired Bootstrap Comparison: Pure Hybrid vs Matched Baseline

A point difference on a single test set can arise by chance. For every classifier, the test set is resampled with replacement 1,000 times. On each resample, the **Pure Hybrid and its matched Baseline** (same classifier, same thresholds) are scored on the *same* participants, which gives a paired distribution of differences.

* The comparison is **matched by classifier**, so it isolates the effect of adding the cluster features.
* The **primary model** for the headline result is the Pure Hybrid with the highest **training OOF F1**. It is selected without looking at the test set.
* A difference is described as statistically supported only if its **95% percentile interval excludes zero**.

These intervals reflect test-set sampling variability only; they do not include variability from retraining the models.

In [ ]:
primary = oof_results[oof_results.Version == "Pure Hybrid"].sort_values("OOF_F1", ascending=False).iloc[0]
PRIMARY_CLF = primary.Classifier
print(f"Primary model (highest training OOF F1 among Pure Hybrids): {PRIMARY_CLF} "
      f"(OOF F1 = {primary.OOF_F1:.3f})")

thr_lookup = oof_results.set_index(["Classifier", "Version"])["Threshold"]
rng = np.random.default_rng(RANDOM_STATE)
y_arr = y_test.to_numpy()
BOOT_METRICS = ["AUROC", "AUPRC", "F1", "Recall"]


def boot_metrics(y, s, t):
    p = (s >= t).astype(int)
    return {"AUROC": roc_auc_score(y, s), "AUPRC": average_precision_score(y, s),
            "F1": f1_score(y, p, zero_division=0), "Recall": recall_score(y, p, zero_division=0)}


boot_rows = []
for b in range(N_BOOTSTRAP):
    idx = rng.integers(0, len(y_arr), len(y_arr))
    yy = y_arr[idx]
    if yy.min() == yy.max():
        continue
    for clf in CLASSIFIERS:
        for version in ["Baseline", "Pure Hybrid"]:
            m = boot_metrics(yy, test_scores[(clf, version)][idx], thr_lookup[(clf, version)])
            boot_rows.append({"Bootstrap": b, "Classifier": clf, "Version": version, **m})
boot = pd.DataFrame(boot_rows)
boot.to_csv(os.path.join(OUT_DIR, "bootstrap_raw.csv"), index=False)

ci_rows = []
for clf in CLASSIFIERS:
    bb = boot[(boot.Classifier == clf) & (boot.Version == "Baseline")].set_index("Bootstrap")
    hh = boot[(boot.Classifier == clf) & (boot.Version == "Pure Hybrid")].set_index("Bootstrap")
    for met in BOOT_METRICS:
        for label, series in [("Baseline", bb[met]), ("Pure Hybrid", hh[met]), ("Difference", hh[met] - bb[met])]:
            ci_rows.append({"Classifier": clf, "Quantity": label, "Metric": met,
                            "Mean": series.mean(), "CI_Lower": series.quantile(0.025), "CI_Upper": series.quantile(0.975)})
bootstrap_ci = pd.DataFrame(ci_rows)
bootstrap_ci["CI_Excludes_0"] = np.where(
    bootstrap_ci.Quantity == "Difference",
    (bootstrap_ci.CI_Lower > 0) | (bootstrap_ci.CI_Upper < 0), np.nan)
bootstrap_ci.to_csv(os.path.join(OUT_DIR, "bootstrap_95ci_pure_vs_baseline.csv"), index=False)

diff_ci = bootstrap_ci[bootstrap_ci.Quantity == "Difference"].reset_index(drop=True)
print(f"Paired bootstrap ({boot.Bootstrap.nunique()} resamples): Pure Hybrid − matched Baseline")
display(diff_ci.round(4))

In [ ]:
fig, axes = plt.subplots(1, len(BOOT_METRICS), figsize=(18, 4.5), sharey=True)
ypos = np.arange(len(CLASSIFIERS))
for ax, met in zip(axes, BOOT_METRICS):
    d = diff_ci[diff_ci.Metric == met].set_index("Classifier").loc[CLASSIFIERS]
    colours = ["#0f766e" if lo > 0 else "#dc2626" if hi < 0 else "#6b7280"
               for lo, hi in zip(d.CI_Lower, d.CI_Upper)]
    for y_i, (m, lo, hi, c) in enumerate(zip(d.Mean, d.CI_Lower, d.CI_Upper, colours)):
        ax.errorbar(m, y_i, xerr=[[m - lo], [hi - m]], fmt="o", color=c, capsize=4)
    ax.axvline(0, color="black", ls="--", lw=0.8)
    ax.set_yticks(ypos, [f"{c} *" if c == PRIMARY_CLF else c for c in CLASSIFIERS])
    ax.set_title(f"Δ {met}")
fig.suptitle("Pure Hybrid − Matched Baseline: Paired Bootstrap 95% CI "
             "(green/red = CI excludes 0; * = primary model selected by training OOF F1)", y=1.04)
save_fig("bootstrap_difference_ci")

# Threshold selection curve and confusion matrices for the primary comparison
table = thr_tables[(PRIMARY_CLF, "Pure Hybrid")]
t_sel = thr_lookup[(PRIMARY_CLF, "Pure Hybrid")]
plt.figure(figsize=(8, 5))
for col, c in [("F1", "#0f766e"), ("Recall", "#2563eb"), ("Precision", "#d97706")]:
    plt.plot(table.Threshold, table[col], label=col, color=c)
plt.axhline(MIN_RECALL, ls=":", color="gray", label=f"Recall floor = {MIN_RECALL}")
plt.axvline(t_sel, ls="--", color="black", label=f"Selected = {t_sel:.2f}")
plt.title(f"OOF Threshold Selection: {PRIMARY_CLF} (Pure Hybrid, training data)")
plt.xlabel("Decision threshold")
plt.ylabel("Training OOF score")
plt.ylim(0, 1)
plt.legend()
save_fig("primary_threshold_selection")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, version in zip(axes, ["Baseline", "Pure Hybrid"]):
    cm = confusion_matrix(y_test, test_preds[(PRIMARY_CLF, version)], labels=[0, 1])
    ax.imshow(cm, cmap="Blues")
    for a in range(2):
        for b in range(2):
            ax.text(b, a, f"{cm[a, b]:,}", ha="center", va="center",
                    color="white" if cm[a, b] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1], ["No CVD", "CVD"])
    ax.set_yticks([0, 1], ["No CVD", "CVD"])
    ax.set(title=f"{PRIMARY_CLF}: {version}", xlabel="Predicted", ylabel="Actual")
    ax.grid(False)
save_fig("primary_confusion_matrices")

## 12. Results Summary

The cell below writes `outputs/RESULTS_SUMMARY.md` from the computed results: the main table, the bootstrap differences, the cluster diagnostics and an automatically worded interpretation. The wording follows fixed rules:

* A Pure Hybrid change is called an **improvement** (or a **decline**) only when the paired 95% CI excludes zero. Otherwise it is reported as **no clear difference**.
* The silhouette coefficient is described using the Kaufman & Rousseeuw scale.
* The Augmented Hybrid is reported as a secondary analysis, because it uses outcome information (via out-of-fold target encoding).

In [ ]:
def md_table(frame, floatfmt="{:.3f}"):
    """Render a DataFrame as a GitHub-flavoured markdown table (no extra dependencies)."""
    cols = list(frame.columns)
    out = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    for _, r in frame.iterrows():
        cells = [floatfmt.format(v) if isinstance(v, (float, np.floating)) else str(v) for v in r]
        out.append("| " + " | ".join(cells) + " |")
    return "\n".join(out)


def verdict(row):
    if row.CI_Lower > 0: return "improvement"
    if row.CI_Upper < 0: return "decline"
    return "no clear difference"


main_table = results[["Classifier", "Version", "Threshold", "Accuracy", "Precision",
                      "Recall_Sensitivity", "Specificity", "F1_Score", "AUROC", "AUPRC", "FN", "FP"]].copy()
main_table[["Classifier", "Version"]] = main_table[["Classifier", "Version"]].astype(str)
main_table["Threshold"] = main_table["Threshold"].map(lambda v: f"{v:.2f}")

diff_view = diff_ci[["Classifier", "Metric", "Mean", "CI_Lower", "CI_Upper"]].copy()
diff_view["Verdict"] = diff_ci.apply(verdict, axis=1)

best_test = results.loc[results.F1_Score.idxmax()]
sil_mean = stability.Silhouette.mean()

lines = []
for clf in CLASSIFIERS:
    d = diff_view[diff_view.Classifier == clf].set_index("Metric")
    parts = [f"{m} {d.loc[m, 'Mean']:+.3f} ({d.loc[m, 'Verdict']})" for m in BOOT_METRICS]
    lines.append(f"- **{clf}**{' (primary)' if clf == PRIMARY_CLF else ''}: " + "; ".join(parts))

n_improved = (diff_view.Verdict == "improvement").sum()
n_declined = (diff_view.Verdict == "decline").sum()
if n_improved and not n_declined:
    overall = "The cluster features gave statistically supported improvements on some classifier–metric pairs and no supported declines."
elif n_declined and not n_improved:
    overall = "The cluster features gave statistically supported declines on some classifier–metric pairs and no supported improvements."
elif n_improved and n_declined:
    overall = "The effect of the cluster features was model-specific: some classifier–metric pairs improved and others declined."
else:
    overall = ("No classifier showed a statistically supported difference between the Pure Hybrid and its baseline; "
               "the cluster features did not consistently improve prediction.")

summary_md = f"""# Results Summary

_Generated automatically by the notebook. Test set: n = {len(y_test):,} (CVD prevalence {y_test.mean() * 100:.2f}%). Training set: n = {len(y_train):,}._

## Clustering
- K = {K_FINAL} (silhouette-optimal K on training data: {best_k}).
- Mean silhouette over {N_STABILITY_SEEDS} seeds: **{sil_mean:.3f}** ({describe_silhouette(sil_mean)}).
- Mean pairwise ARI across seeds: **{np.mean(aris):.3f}** (min {np.min(aris):.3f}).

## Held-out test performance (all 12 models)
Thresholds were selected from training out-of-fold predictions (maximise F1 subject to recall >= {MIN_RECALL}).

{md_table(main_table)}

## Pure Hybrid minus matched Baseline (paired bootstrap, {boot.Bootstrap.nunique()} resamples)
{md_table(diff_view, "{:+.4f}")}

## Interpretation
- Primary model (selected by training OOF F1, not by test results): **{PRIMARY_CLF} – Pure Hybrid**.
- Highest test F1 among the 12 models: {best_test.Classifier} – {best_test.Version} (F1 = {best_test.F1_Score:.3f}). It is reported for information only and was not used for model selection.
{chr(10).join(lines)}
- {overall}
- Augmented Hybrid results (with `Cluster_CVD_Rate`) are a secondary analysis and should not be presented as the main hybrid result.
"""

with open(os.path.join(OUT_DIR, "RESULTS_SUMMARY.md"), "w", encoding="utf-8") as f:
    f.write(summary_md)
print(summary_md)
print("All tables saved to", OUT_DIR, "and figures to", FIG_DIR)

## How to Report These Results

1. **Use `outputs/results_all_12_models.csv` as the single source of final metrics.** Do not mix them with numbers from earlier notebook versions.
2. **Present the Pure Hybrid as the main hybrid model.** The Augmented Hybrid uses outcome-derived information and is a secondary sensitivity analysis.
3. **Only claim an improvement when the paired bootstrap 95% CI excludes zero.** Otherwise use wording such as "no clear difference", "small model-specific change" or "no consistent improvement across classifiers".
4. **Describe cluster separation with the silhouette scale printed above.** Low silhouette values mean overlapping clusters, even if the cluster profiles are clinically interpretable.
5. **Thresholds and the primary model were chosen from training data only.** State this explicitly, because it is what makes the test results unbiased.

## Limitations

* **Survey design not modelled.** NHANES uses a complex, multistage probability sample. Survey weights, strata and PSUs are not used here, so prevalence and performance estimates describe this analytic sample and are **not nationally representative** estimates for the US population.
* **Cross-sectional data.** CVD status and the predictors are measured at the same time. The models identify participants who *have* CVD; they do not predict *future* events, and some predictors may be affected by treatment after diagnosis.
* **Outcome definition.** The CVD label is taken as provided in the prepared dataset (see `data/README.md`). If it is based on self-report, it is subject to recall and diagnosis bias.
* **SMOTE and categorical variables.** SMOTE interpolates in the one-hot encoded space, which can create synthetic samples with fractional category values. SMOTENC is a possible alternative.
* **Fixed hyperparameters.** Hyperparameters were not tuned, so that all representations are compared under identical settings. Absolute performance could improve with nested tuning.
* **Bootstrap scope.** The confidence intervals reflect test-set sampling variability, not variability due to retraining on different samples.
* **Single dataset, no external validation.** The results have not been validated on an independent cohort and are not suitable for clinical use.